# foreachBatch Patterns
**Topic**: Streaming | **Exercises**: 7 | **Total Time**: ~95 min

Practice `foreachBatch`, the escape hatch that lets you apply arbitrary batch DataFrame
logic to each streaming microbatch. Covers basic appends, MERGE in streaming, multi-sink
writes, `batch_id` for idempotency, quality routing to a quarantine table, running
microbatch metrics, and materializing `batch_df` via `collect()` when it must be read more than once.

**Solutions**: If stuck, see `solutions/foreachbatch-patterns-solutions.py` for hints and answers.

**Key concepts**:
- `foreachBatch(fn)` calls `fn(batch_df, batch_id)` once per microbatch with a static DataFrame
- Inside the function you can use any batch API: MERGE, multi-sink writes, joins, materialization via `collect()`
- Every stream uses `.trigger(availableNow=True)` + `.awaitTermination()` so the assertion cell runs only after the stream finishes
- Sources are Delta tables (Free Edition has no Kafka)
- To make the function idempotent across retries, you must dedupe on `batch_id` or rely on a transactional sink (MERGE on a primary key is transactional, plain append is not)

**Free Edition note: caching workaround inside foreachBatch**

Several exercises below (Ex 3, 5, 6, 7) read `batch_df` more than once inside the
`foreachBatch` function. On classic compute the standard fix is `batch_df.cache()` +
`unpersist()` so both reads share one materialization. On Databricks Free Edition
(serverless), `cache()` / `persist()` is rejected with `[NOT_SUPPORTED_WITH_SERVERLESS]`.
The serverless-safe equivalent is to materialize the batch once via `collect()` and rebuild
a local DataFrame from the rows:

```python
rows = batch_df.collect()
materialized = batch_df.sparkSession.createDataFrame(rows, batch_df.schema)
# then read `materialized` as many times as you need
```

Use this pattern in every multi-read foreachBatch on Free Edition. Each affected exercise
below points back to this note.

**Setup complete.** All exercise tables are in `db_code.foreachbatch_patterns`.

**Source tables**: `ex1_source` ... `ex7_source` are identical copies of a 40-row event stream:
- 36 "good" rows (`event_id` EV-301..EV-336, valid `amount`, valid `user_id`)
- 4 "bad" rows (EV-901/902 have negative `amount`; EV-903/904 have NULL `user_id`)

**Schema** (`exN_source`):
| Column     | Type      | Notes |
|------------|-----------|-------|
| event_id   | STRING    | Unique within source |
| event_type | STRING    | purchase / view / click |
| user_id    | STRING    | Nullable; 2 rows have NULL |
| amount     | DOUBLE    | Nullable; 2 rows are negative |
| event_ts   | TIMESTAMP | Event timestamp |

**Checkpoints**: Use the `CHECKPOINT_BASE` variable (`/Volumes/db_code/foreachbatch_patterns/checkpoints/`).
Each exercise must use its own subdirectory (e.g., `f"{CHECKPOINT_BASE}/ex1"`).

**Pattern**: Each exercise reads from `db_code.foreachbatch_patterns.exN_source` as a stream and writes via a `foreachBatch` function to `exN_target` (or, for Exercise 3, to both `ex3_target_raw` and `ex3_target_clean`).